In [1]:
import json
import os.path
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import pickle
import scienceplots

In [2]:
kitgreen="#009682"
green="#98BF64"
kitblue="#4664AA"
grey5="#f2f2f2ff"
grey30="#b3b3b3ff"

# British Data Overview and Preparation
# 英国数据总览与准备

## 地理数据

In [3]:
# Read the geographic boundary data of UK Local Authority Districts (LAD) (.shp file)
LAD_region = gpd.read_file(r"./dataset/HDRah-Data-PS-GB-1f63a32/Geo_data/LAD_DEC_2022_UK_BFC.shp", encoding='ISO-8859-1')
LAD_region = LAD_region.to_crs(epsg=4326)
LAD_region = LAD_region[["LAD22CD", "LAD22NM", "geometry"]]

In [4]:
# Read the mapping data of UK Local Authority Districts (LAD) to Integrated Territorial Units for Statistics (ITL) regions
region_mapping = pd.read_csv(r"./dataset/Local_Authority_District_(April_2021)_to_LAU1_to_ITL3_to_ITL2_to_ITL1_(January_2021)_Lookup_in_United_Kingdom.csv")
region_mapping = region_mapping[["LAD21CD", "LAD21NM", "ITL321CD", "ITL221CD"]]

In [5]:
# Add ITL (Integrated Territorial Units for Statistics) region codes via a mapping table
LAD_region["ITL3"] = LAD_region["LAD22CD"].apply(lambda x: region_mapping.loc[region_mapping["LAD21CD"]==x, "ITL321CD"].values[0])
LAD_region["ITL2"] = LAD_region["LAD22CD"].apply(lambda x: region_mapping.loc[region_mapping["LAD21CD"]==x, "ITL221CD"].values[0])
LAD_region.rename(columns={"LAD22CD": "LAD", "LAD22NM": "LADName"}, inplace=True)

In [6]:
LAD_region

,LAD,LADName,geometry,ITL3,ITL2
0,E06000001,Hartlepool,"MULTIPOLYGON (((-1.22469 54.6261, -1.22491 54....",TLC11,TLC1
1,E06000002,Middlesbrough,"MULTIPOLYGON (((-1.27719 54.54783, -1.27719 54...",TLC12,TLC1
2,E06000003,Redcar and Cleveland,"MULTIPOLYGON (((-1.20097 54.57762, -1.20029 54...",TLC12,TLC1
3,E06000004,Stockton-on-Tees,"MULTIPOLYGON (((-1.27209 54.55336, -1.27211 54...",TLC11,TLC1
4,E06000005,Darlington,"POLYGON ((-1.63767 54.61713, -1.63766 54.61669...",TLC13,TLC1
...,...,...,...,...,...
369,W06000020,Torfaen,"POLYGON ((-3.10489 51.79504, -3.10171 51.79372...",TLL16,TLL1
370,W06000021,Monmouthshire,"MULTIPOLYGON (((-2.78124 51.5253, -2.78142 51....",TLL21,TLL2
371,W06000022,Newport,"MULTIPOLYGON (((-2.95222 51.62897, -2.95199 51...",TLL21,TLL2
372,W06000023,Powys,"MULTIPOLYGON (((-3.9119 52.56284, -3.90951 52....",TLL24,TLL2


## 人口以及GVA数据

In [7]:
# Land Use Distribution
# url: https://www.iea.org/countries/united-kingdom/efficiency-demand
uk_energy_consumption_share = {
    "transportation": 0.3410,
    "residential": 0.2792,
    "industrial": 0.1794,
    "commercial": 0.1397,
    "agricultural": 0.0105,
    "others": 0.0503
}

# GVA 经济活动分类 (SIC07 classification for GVA) from LLM
sic07_classification = {
    "industrial_gva": ["C (10-33)", "DE (35-39)", "F (41-43)"],
    "commercial_gva": ["G (45-47)","H (49-53)", "I (55-56)", "J (58-63)", "K (64-66)", "L (68)"],
    "agricultural_gva": ["AB (1-9)"],
    "others_gva": ["M (69-75)", "N (77-82)", "O (84)", "P (85)", "Q (86-88)", "R (90-93)", "S (94-96)", "T (97-98)"]
}

In [8]:
# Loading population data
# url: https://www.ons.gov.uk/peoplepopulationandcommunity/populationandmigration/populationestimates/datasets/estimatesofthepopulationforenglandandwales
pop = pd.read_excel(r"./dataset/mye23tablesew.xlsx", sheet_name="MYE5", skiprows=7)
pop = pop[["Code", "Estimated Population mid-2023"]].set_index("Code")

In [9]:
LAD_region = LAD_region.merge(pop, left_on='LAD', right_on='Code', how='left')
LAD_region.rename(columns={'Estimated Population mid-2023': 'population'}, inplace=True)

In [10]:
nan_population_rows = LAD_region[LAD_region['population'].isnull()]
itl2_to_remove = nan_population_rows['ITL2'].unique()
print(f"检测到人口数据缺失，将剔除以下ITL2区域的所有相关行: {list(itl2_to_remove)}")
filtered_gdf = LAD_region[~LAD_region['ITL2'].isin(itl2_to_remove)]

检测到人口数据缺失，将剔除以下ITL2区域的所有相关行: ['TLD1', 'TLE2', 'TLK2', 'TLN0', 'TLM7', 'TLM9', 'TLM8', 'TLM6', 'TLM5']


In [11]:
# 2a. Aggregate the attributes using pandas groupby
aggregation_functions = {'population': 'sum', 'ITL2': 'first'}
aggregated_attributes = filtered_gdf.drop(columns='geometry').groupby('ITL3').agg(aggregation_functions)
# 2b. Reset index to make 'ITL3' a column again
dissolved_geometry = filtered_gdf[['ITL3', 'geometry']].dissolve(by='ITL3')

In [12]:
ITL3_region = dissolved_geometry.join(aggregated_attributes)

In [13]:
total_england_population = ITL3_region.population.sum()
ITL3_region["residential_percent"] = ITL3_region["population"] / total_england_population

In [14]:
ITL3_region

,geometry,population,ITL2,residential_percent
ITL3,,,,
TLC11,"MULTIPOLYGON (((-1.25065 54.62529, -1.25063 54...",297781.0,TLC1,0.005121
TLC12,"MULTIPOLYGON (((-1.24878 54.59065, -1.2484 54....",290588.0,TLC1,0.004998
TLC13,"POLYGON ((-1.63767 54.61713, -1.63766 54.61669...",110562.0,TLC1,0.001901
TLC14,"POLYGON ((-1.73702 54.91846, -1.73695 54.91845...",532182.0,TLC1,0.009153
TLC21,"MULTIPOLYGON (((-2.02784 55.76838, -2.02759 55...",327055.0,TLC2,0.005625
...,...,...,...,...
TLL18,"MULTIPOLYGON (((-4.27222 51.55578, -4.27211 51...",246742.0,TLL1,0.004244
TLL21,"MULTIPOLYGON (((-2.78154 51.5257, -2.78129 51....",258200.0,TLL2,0.004441
TLL22,"MULTIPOLYGON (((-3.12101 51.38002, -3.1209 51....",518269.0,TLL2,0.008913


In [15]:
# https://www.ons.gov.uk/economy/grossvalueaddedgva/datasets/nominalandrealregionalgrossvalueaddedbalancedbyindustry/current
gva_raw = pd.read_excel(r"./dataset/regionalgrossvalueaddedbalancedbyindustryandallitlregions.xlsx", sheet_name="Table 3c", skiprows=1)
gva_pivot = gva_raw.pivot_table(index='ITL code', columns='SIC07 code', values='2022', aggfunc='sum')

# 按分类聚合GVA (Aggregate GVA by category)
gva_by_category = pd.DataFrame(index=gva_pivot.index)
for category, codes in sic07_classification.items():
    # 确保所有代码都存在于列中，避免KeyError
    valid_codes = [code for code in codes if code in gva_pivot.columns]
    gva_by_category[category] = gva_pivot[valid_codes].sum(axis=1)

In [16]:
gva_by_category

,industrial_gva,commercial_gva,agricultural_gva,others_gva
ITL code,,,,
TLC11,2024,2857,26,2628
TLC12,969,2073,125,2314
TLC13,268,1378,26,1040
TLC14,4042,2979,135,3498
TLC21,1883,2175,338,1903
...,...,...,...,...
TLN0C,456,1073,147,828
TLN0D,982,1608,56,1100
TLN0E,1079,1293,59,1475


In [17]:
# total_gva = gva_by_category.sum().sum()
# gva_by_category = gva_by_category / total_gva
# gva_by_category

In [18]:
gva_by_category["industrial_percent"] = gva_by_category["industrial_gva"]/ gva_by_category["industrial_gva"].sum()
gva_by_category["commercial_percent"] = gva_by_category["commercial_gva"]/ gva_by_category["commercial_gva"].sum()
gva_by_category["agricultural_percent"] = gva_by_category["agricultural_gva"]/ gva_by_category["agricultural_gva"].sum()
gva_by_category["others_percent"] = gva_by_category["others_gva"]/ gva_by_category["others_gva"].sum()

In [19]:
ITL3_region = ITL3_region.merge(gva_by_category, left_on='ITL3', right_index=True, how='left')
ITL3_region.reset_index(inplace=True)

In [20]:
# Calculate the area percentage of each LAD within its ITL region
ITL3_region = ITL3_region.to_crs(epsg=3857)
ITL3_region["area"] = ITL3_region.geometry.area
area_by_region = ITL3_region.groupby('ITL2')['area'].sum()
ITL3_region['area_percent'] = ITL3_region.apply(lambda row: (row['area'] / area_by_region[row['ITL2']]), axis=1)
ITL3_region = ITL3_region.to_crs(epsg=4326)

In [21]:
ITL3_region

,ITL3,geometry,population,ITL2,residential_percent,industrial_gva,commercial_gva,agricultural_gva,others_gva,industrial_percent,commercial_percent,agricultural_percent,others_percent,area,area_percent
0,TLC11,"MULTIPOLYGON (((-1.25065 54.62529, -1.25063 54...",297781.0,TLC1,0.005121,2024,2857,26,2628,0.005072,0.002821,0.001086,0.003307,8.884548e+08,0.098356
1,TLC12,"MULTIPOLYGON (((-1.24878 54.59065, -1.2484 54....",290588.0,TLC1,0.004998,969,2073,125,2314,0.002428,0.002047,0.005223,0.002912,8.873307e+08,0.098231
2,TLC13,"POLYGON ((-1.63767 54.61713, -1.63766 54.61669...",110562.0,TLC1,0.001901,268,1378,26,1040,0.000672,0.001361,0.001086,0.001309,5.861505e+08,0.064889
3,TLC14,"POLYGON ((-1.73702 54.91846, -1.73695 54.91845...",532182.0,TLC1,0.009153,4042,2979,135,3498,0.010130,0.002941,0.005641,0.004402,6.671151e+09,0.738524
4,TLC21,"MULTIPOLYGON (((-2.02784 55.76838, -2.02759 55...",327055.0,TLC2,0.005625,1883,2175,338,1903,0.004719,0.002148,0.014124,0.002395,1.545685e+10,0.904344
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,TLL18,"MULTIPOLYGON (((-4.27222 51.55578, -4.27211 51...",246742.0,TLL1,0.004244,650,2490,19,2613,0.001629,0.002459,0.000794,0.003288,9.793416e+08,0.027847
134,TLL21,"MULTIPOLYGON (((-2.78154 51.5257, -2.78129 51....",258200.0,TLL2,0.004441,1565,2441,58,2578,0.003922,0.002410,0.002424,0.003244,2.710832e+09,0.132648
135,TLL22,"MULTIPOLYGON (((-3.12101 51.38002, -3.1209 51....",518269.0,TLL2,0.008913,2388,7551,26,6252,0.005985,0.007456,0.001086,0.007867,1.218979e+09,0.059648
136,TLL23,"MULTIPOLYGON (((-2.96259 53.13256, -2.96237 53...",291961.0,TLL2,0.005021,4070,2017,180,2458,0.010200,0.001992,0.007522,0.003093,2.613273e+09,0.127874


# Substation Data

In [22]:
# Read UK substation data (including location, demand, capacity, etc.)
substations = gpd.read_file(r"./dataset/HDRah-Data-PS-GB-1f63a32/GB_PS_data_extend.csv")
substations["Geo(Long,Lat)"] = substations["Geo(Long,Lat)"].apply(lambda x: [float(coord) for coord in x.split(',')])
substations["geometry"] = substations.apply(lambda x: Point(x["Geo(Long,Lat)"][0], x["Geo(Long,Lat)"][1]), axis=1)
substations = substations.set_geometry("geometry")
substations = substations.set_crs(epsg=4326, inplace=True)

In [23]:
# Aggregate substations at the same location (sum numeric values, take the first for others)
numeric_columns = substations.select_dtypes(include=np.number).columns.tolist()
non_numeric_columns = substations.select_dtypes(exclude=np.number).columns.tolist()

# Create aggregation dictionary: 'sum' for numeric columns, 'first' for non-numeric columns
agg_dict = {}
for col in numeric_columns:
    agg_dict[col] = 'sum'
for col in non_numeric_columns:
    agg_dict[col] = 'first'

del agg_dict["geometry"] # 几何列不参与聚合
# Group and aggregate
substations = substations.groupby("geometry", as_index=False).agg(agg_dict)
substations = gpd.GeoDataFrame(substations, geometry='geometry') # 转换回GeoDataFrame

In [24]:
# Data cleaning and formatting
substations["Demand (MVA)"] = pd.to_numeric(substations["Demand (MVA)"], errors='coerce')
substations["Firm Capacity (MVA)"] = pd.to_numeric(substations["Firm Capacity (MVA)"], errors='coerce')
substations["RegName"] = substations["RegName"].str.replace(" ", "")
substations = substations[["PS Name", "Demand (MVA)", "Firm Capacity (MVA)", "RegName", "geometry", "RegID"]]

In [25]:
# Use spatial join to assign each substation to its corresponding LAD
substation_regions = gpd.sjoin(substations, ITL3_region, how="left", predicate="within")
substations["ITL3"] = substation_regions["ITL3"]
substations["ITL2"] = substation_regions["ITL2"]

In [26]:
# Aggregate total demand and capacity of substations by itl3 region
region_stats = substation_regions.groupby(["ITL3"]).agg({"Demand (MVA)": "sum","Firm Capacity (MVA)": "sum"}).reset_index()
ITL3_region = ITL3_region.merge(region_stats, on=["ITL3"], how="left")

In [27]:
ITL3_region.loc[ITL3_region['ITL2'].isin(["TLI3", "TLI4", "TLI5", "TLI6", "TLI7"]) , 'ITL2'] = "London"
substations.loc[substations['ITL2'].isin(["TLI3", "TLI4", "TLI5", "TLI6", "TLI7"]) , 'ITL2'] = "London"

In [28]:
ITL3_region

,ITL3,geometry,population,ITL2,residential_percent,industrial_gva,commercial_gva,agricultural_gva,others_gva,industrial_percent,commercial_percent,agricultural_percent,others_percent,area,area_percent,Demand (MVA),Firm Capacity (MVA)
0,TLC11,"MULTIPOLYGON (((-1.25065 54.62529, -1.25063 54...",297781.0,TLC1,0.005121,2024,2857,26,2628,0.005072,0.002821,0.001086,0.003307,8.884548e+08,0.098356,217.451304,350.20
1,TLC12,"MULTIPOLYGON (((-1.24878 54.59065, -1.2484 54....",290588.0,TLC1,0.004998,969,2073,125,2314,0.002428,0.002047,0.005223,0.002912,8.873307e+08,0.098231,152.654033,215.81
2,TLC13,"POLYGON ((-1.63767 54.61713, -1.63766 54.61669...",110562.0,TLC1,0.001901,268,1378,26,1040,0.000672,0.001361,0.001086,0.001309,5.861505e+08,0.064889,80.803301,150.00
3,TLC14,"POLYGON ((-1.73702 54.91846, -1.73695 54.91845...",532182.0,TLC1,0.009153,4042,2979,135,3498,0.010130,0.002941,0.005641,0.004402,6.671151e+09,0.738524,357.172666,722.20
4,TLC21,"MULTIPOLYGON (((-2.02784 55.76838, -2.02759 55...",327055.0,TLC2,0.005625,1883,2175,338,1903,0.004719,0.002148,0.014124,0.002395,1.545685e+10,0.904344,214.998826,445.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,TLL18,"MULTIPOLYGON (((-4.27222 51.55578, -4.27211 51...",246742.0,TLL1,0.004244,650,2490,19,2613,0.001629,0.002459,0.000794,0.003288,9.793416e+08,0.027847,167.290000,297.10
134,TLL21,"MULTIPOLYGON (((-2.78154 51.5257, -2.78129 51....",258200.0,TLL2,0.004441,1565,2441,58,2578,0.003922,0.002410,0.002424,0.003244,2.710832e+09,0.132648,193.490000,301.00
135,TLL22,"MULTIPOLYGON (((-3.12101 51.38002, -3.1209 51....",518269.0,TLL2,0.008913,2388,7551,26,6252,0.005985,0.007456,0.001086,0.007867,1.218979e+09,0.059648,334.170000,662.80
136,TLL23,"MULTIPOLYGON (((-2.96259 53.13256, -2.96237 53...",291961.0,TLL2,0.005021,4070,2017,180,2458,0.010200,0.001992,0.007522,0.003093,2.613273e+09,0.127874,266.405791,494.20


In [29]:
substations

,PS Name,Demand (MVA),Firm Capacity (MVA),RegName,geometry,RegID,ITL3,ITL2
0,Isles Of Scilly,3.74000,5.50,SouthWest,POINT (-6.3097 49.914),E12000009,TLK30,TLK3
1,Brawdy,4.23000,14.00,Pembrokeshire,POINT (-5.0965 51.8873),W92000004,TLL14,TLL1
2,St Davids,1.56000,2.00,Pembrokeshire,POINT (-5.2534 51.882),W92000004,TLL14,TLL1
3,Geevor,4.59000,10.50,SouthWest,POINT (-5.6739 50.1485),E12000009,TLK30,TLK3
4,Mousehole,2.45000,4.13,SouthWest,POINT (-5.5484 50.084),E12000009,TLK30,TLK3
...,...,...,...,...,...,...,...,...
4171,wittersham,1.40000,0.80,SouthEast,POINT (0.7216 51.02658),E12000008,TLJ45,TLJ4
4172,tenterden,6.94434,8.70,SouthEast,POINT (0.67531 51.07763),E12000008,TLJ45,TLJ4
4173,rye,8.49954,9.80,SouthEast,POINT (0.76066 50.95534),E12000008,TLJ22,TLJ2
4174,kenardington,3.04340,2.00,SouthEast,POINT (0.79018 51.07581),E12000008,TLJ45,TLJ4


In [30]:
ITL3_region.to_file("./results/intermediate/ITL3_region.gpkg", driver='GPKG')
substations.to_file("./results/intermediate/substations.gpkg", driver='GPKG')

# Plotting the regions and substations

In [31]:
interested_locations = {}
unique_itl2_regions = ITL3_region['ITL2'].unique()
for itl2_code in interested_locations.keys():
    interested_region = ITL3_region[ITL3_region['ITL2'] == itl2_code]
    interested_substations = substations[substations['ITL2'] == itl2_code]

    interested_locations[itl2_code] = {
        "region": interested_region,
        "substation": interested_substations
    }

In [32]:
# for i, key in enumerate(interested_locations.keys()):
#     print(key)
#     fig, ax = plt.subplots(1, 1, figsize=(3.5, 2.5), dpi=300)  # 双栏宽度
#     interested_locations[key]["region"].plot(color=grey5, edgecolor=grey30, figsize=(7, 5), ax=ax, linewidth = 0.5, aspect='equal')
#     interested_locations[key]["substation"].plot(ax=ax, color=kitblue, markersize=0.2, aspect='equal')
#     # for x, y, label in zip(Ethos_regions.geometry.centroid.x, Ethos_regions.geometry.centroid.y, Ethos_regions['index']):
#     #     ax.text(x, y, label, fontsize=4, ha='center', va='center')
#     ax.tick_params(axis='x', labelsize=6)  # 调整x轴刻度标签的大小
#     ax.tick_params(axis='y', labelsize=6)
#     ax.set_xlabel(r'Longitude [$^\circ$E]', fontsize=8)
#     ax.set_ylabel(r'Latitude [$^\circ$N]', fontsize=8)
#     # ax.set_title('Original Surfaces and Points', fontsize=8)
#     plt.tight_layout()
#     # plt.savefig(f'./results/pictures/Figure_interested_region_{key}.eps', dpi=300, bbox_inches='tight')
#     plt.show()